# Numerical Issues in Transformers

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 06.06 |
| Time | ~70 min |
| Colab GPU | Needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/07_numerical_issues_in_transformers.ipynb)

---

## 🎯 Learning Objective

Diagnose attention-score overflow and residual-stream magnitude growth, and compare pre-norm vs. post-norm placement for training stability.

---

## 📐 Theory

This capstone-style notebook for Module 06 applies every earlier idea — IEEE 754 (`06.01`),
stable softmax (`06.02`), and mixed precision (`06.06`) — to the specific architecture where
they matter most in practice: the transformer.

### Attention score overflow

Self-attention computes $\text{softmax}(QK^\top/\sqrt{d_k})V$ (recall `01.10`). The scores
$QK^\top/\sqrt{d_k}$ are unbounded — nothing in the formula caps their magnitude — and if $Q$,
$K$ carry large activations (common before the network's normalization layers have had a
chance to constrain them, especially early in training or in poorly-initialized layers), the
scores can reach magnitudes in the hundreds or thousands. Even when those scores are perfectly
representable in a given float format (recall `06.01`), $\exp(\text{score})$ inside softmax
overflows far sooner: $e^{89}$ already exceeds float32's maximum representable value, and
float16 overflows even earlier. This is exactly `06.02`'s stable-softmax problem, reappearing
in a specific, load-bearing spot inside every transformer layer — which is why production
attention implementations always use the numerically stable (subtract-the-max) softmax formula,
never the naive one.

### Pre-norm vs. post-norm placement

The original Transformer places LayerNorm *after* each sublayer, applied to the sum of the
residual and the sublayer's output — **post-norm**:
$\mathbf{h} \leftarrow \text{LayerNorm}(\mathbf{h} + \text{Sublayer}(\mathbf{h}))$. Later
architectures (GPT-2 onward) commonly use **pre-norm** instead: normalize *before* the sublayer,
and add its output to the *un-normalized* residual stream:
$\mathbf{h} \leftarrow \mathbf{h} + \text{Sublayer}(\text{LayerNorm}(\mathbf{h}))$.

These look like a minor reshuffling, but they have a real, measurable effect connecting
directly to `02.10`'s vanishing/exploding gradient analysis. In post-norm, every layer's output
gets renormalized to a controlled scale, but the identity path (recall `02.10`'s residual
connection discussion) then runs *through* that normalization rather than around it — so the
clean "local gradient exactly $1$" property of a pure residual connection is partially
disrupted by the normalization's own Jacobian. In pre-norm, the residual stream itself is never
directly normalized — each sublayer's contribution is added to a running, unconstrained sum,
which keeps a cleaner identity gradient path but lets the residual stream's *magnitude* grow
steadily with depth (each new layer just adds more to an already-large running total).

### Residual stream magnitude growth

Because pre-norm's residual path is never itself normalized, tracking the residual stream's
norm layer-by-layer typically shows it growing with depth — visible directly in the
Implementation section below. This isn't necessarily a problem on its own (LayerNorm inside
each sublayer keeps the *sublayer's own* computation well-scaled regardless of the running
residual's size), but it does mean later layers see inputs of a different scale than earlier
ones, a real practical consideration when choosing initialization and learning rates for very
deep transformers.

### Why practitioners care about this specific tradeoff

Empirically, pre-norm transformers tend to train more stably at initialization and support
learning-rate warmup with less careful tuning (part of why they became the more common choice
for large modern LLMs), while post-norm can require more careful warmup schedules (recall
`04.06`) to avoid early instability, but has sometimes shown better final performance when
training succeeds. Neither is a universal "correct" answer — this is a real architectural
tradeoff practitioners still navigate, not settled math.

---

In [ ]:
# Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Feeding deliberately large-magnitude Q/K vectors into an attention computation shows naive
softmax collapsing into NaN once $\exp(\text{score})$ overflows float16's range, while the
numerically stable softmax formula stays a valid probability distribution throughout.

In [ ]:
torch.manual_seed(0)
seq_len, d_k = 8, 64

def naive_softmax(scores):
    e = torch.exp(scores)
    return e / e.sum(dim=-1, keepdim=True)

scale_factors = np.linspace(1, 35, 25)
naive_has_nan, stable_has_nan = [], []
for s in scale_factors:
    Q = (torch.randn(seq_len, d_k) * s).half()
    K = (torch.randn(seq_len, d_k) * s).half()
    scores = (Q @ K.T) / np.sqrt(d_k)
    with torch.no_grad():
        naive_probs = naive_softmax(scores)
        stable_probs = torch.softmax(scores, dim=-1)
    naive_has_nan.append(torch.isnan(naive_probs).any().item())
    stable_has_nan.append(torch.isnan(stable_probs).any().item())

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(scale_factors, naive_has_nan, 'o-', color="#C44E52", label="naive softmax (has NaN)")
ax.plot(scale_factors, stable_has_nan, 's-', color="#4C72B0", label="stable softmax (has NaN)")
ax.set_xlabel("Q/K magnitude scale factor"); ax.set_ylabel("any NaN in attention weights? (1=yes)")
ax.set_yticks([0, 1]); ax.set_yticklabels(["No", "Yes"])
ax.set_title("Naive attention softmax breaks down in float16\nwell before the stable version does")
ax.legend()
plt.show()

first_naive_nan = scale_factors[np.argmax(naive_has_nan)] if any(naive_has_nan) else None
print(f"Naive softmax first shows NaN around Q/K scale ~ {first_naive_nan:.1f}")
print(f"Stable softmax never showed NaN across the entire tested range: {not any(stable_has_nan)}")

## 🐍 Implementation from Scratch

We build a minimal attention mechanism with both a naive and a numerically stable softmax,
confirm the exact failure mode with real numbers, then build both pre-norm and post-norm
transformer blocks and track how residual-stream magnitude and gradients behave through depth.

In [ ]:
def attention(Q, K, V, use_stable_softmax=True):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / np.sqrt(d_k)
    if use_stable_softmax:
        probs = torch.softmax(scores, dim=-1)
    else:
        e = torch.exp(scores)
        probs = e / e.sum(dim=-1, keepdim=True)
    return probs @ V, scores

torch.manual_seed(0)
seq_len, d_k = 8, 64
scale = 30  # large enough to push scores past exp()'s float16 overflow point
Q = (torch.randn(seq_len, d_k) * scale).half()
K = (torch.randn(seq_len, d_k) * scale).half()
V = torch.randn(seq_len, d_k).half()

out_stable, scores = attention(Q, K, V, use_stable_softmax=True)
out_naive, _ = attention(Q, K, V, use_stable_softmax=False)
print(f"Score range: [{scores.min().item():.1f}, {scores.max().item():.1f}]  (well within float16's representable range)")
print(f"Naive attention output has NaN:  {torch.isnan(out_naive).any().item()}")
print(f"Stable attention output has NaN: {torch.isnan(out_stable).any().item()}")

# --- Pre-norm vs post-norm: track residual-stream magnitude AND gradient through depth ---
def make_block(d_model):
    ln1, ln2 = torch.nn.LayerNorm(d_model), torch.nn.LayerNorm(d_model)
    attn_proj = torch.nn.Linear(d_model, d_model)
    ff = torch.nn.Sequential(torch.nn.Linear(d_model, d_model * 2), torch.nn.ReLU(), torch.nn.Linear(d_model * 2, d_model))
    return ln1, ln2, attn_proj, ff

def run_stack(n_layers, d_model, pre_norm, seed=0, track_grad=False):
    torch.manual_seed(seed)
    dtype = torch.float64 if track_grad else torch.float32
    x = torch.randn(4, d_model, dtype=dtype, requires_grad=track_grad)
    blocks = [make_block(d_model) for _ in range(n_layers)]
    if track_grad:
        for b in blocks:
            for m in b:
                m.double()
    h, magnitudes = x, [x.norm(dim=-1).mean().item()]
    for ln1, ln2, attn_proj, ff in blocks:
        if pre_norm:
            h = h + attn_proj(ln1(h))
            h = h + ff(ln2(h))
        else:
            h = ln1(h + attn_proj(h))
            h = ln2(h + ff(h))
        magnitudes.append(h.norm(dim=-1).mean().item())
    if track_grad:
        probe = torch.randn(d_model, dtype=dtype)  # generic linear functional; avoids the
        (h * probe).sum().backward()                # trivial-zero artifact of summing a mean-zero LN output
        return magnitudes, x.grad.norm().item()
    return magnitudes, None

d_model = 32
for n_layers in [5, 10, 20, 40]:
    mags_pre, grad_pre = run_stack(n_layers, d_model, pre_norm=True, track_grad=True)
    mags_post, grad_post = run_stack(n_layers, d_model, pre_norm=False, track_grad=True)
    print(f"\nn_layers={n_layers}: pre-norm  final magnitude={mags_pre[-1]:.2f}, input grad norm={grad_pre:.3f}")
    print(f"              post-norm final magnitude={mags_post[-1]:.2f}, input grad norm={grad_post:.3f}")

## 🤖 Why This Matters for AI

Every production transformer implementation — GPT-family models, BERT, LLaMA, the attention
kernel inside PyTorch's `scaled_dot_product_attention` — uses a numerically stable softmax
internally for exactly the reason demonstrated above, and virtually every modern large model
uses pre-norm placement specifically because it tolerates the aggressive learning rates and
minimal warmup tuning that pre-training at scale requires, even though it means accepting
residual-stream growth as a tradeoff. Below, we visualize the magnitude and gradient trends
from the Implementation section together, then connect the residual-stream growth finding back
to `06.01`'s bfloat16-vs-float16 discussion: a growing residual stream is one more concrete
reason transformer training prefers bfloat16's wide exponent range over float16's narrow one.

In [ ]:
n_layers_plot = 40
mags_pre, _ = run_stack(n_layers_plot, d_model, pre_norm=True)
mags_post, _ = run_stack(n_layers_plot, d_model, pre_norm=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(mags_pre, color="#4C72B0", lw=2, label="pre-norm (residual stream grows)")
ax.plot(mags_post, color="#C44E52", lw=2, label="post-norm (magnitude stays controlled)")
ax.set_xlabel("layer depth"); ax.set_ylabel("mean residual-stream magnitude")
ax.set_title("Pre-norm's residual stream accumulates with depth;\npost-norm's stays roughly constant")
ax.legend()
plt.show()

print(f"After {n_layers_plot} layers: pre-norm magnitude={mags_pre[-1]:.1f}, post-norm magnitude={mags_post[-1]:.1f}")

# Does that growing magnitude survive representation in float16 vs bfloat16?
final_stream_pre = torch.full((1,), mags_pre[-1], dtype=torch.float32)
print(f"\nFinal pre-norm residual-stream magnitude, cast to float16:  {final_stream_pre.half().item()}")
print(f"Final pre-norm residual-stream magnitude, cast to bfloat16: {final_stream_pre.bfloat16().item()}")
print("At THIS magnitude both formats are fine -- but recall from 06.01 that float16 overflows")
print("entirely above 65504, while bfloat16 shares float32's full exponent range. A deep enough")
print("pre-norm stack (or one with less-controlled initialization) can push past that boundary")
print("in float16 well before bfloat16 would ever notice -- one more concrete reason bfloat16")
print("is the standard choice for training, not just an arbitrary alternative to float16.")

## 🏋️ Exercises

### Warm-up
1. For $Q, K$ with $d_k=64$ and every entry equal to $c$, compute the exact value of every
   entry of $QK^\top/\sqrt{d_k}$ symbolically in terms of $c$. Find the smallest integer $c$
   that makes $\exp$ of that score overflow float16 (recall float16's max is about $65504$, so
   overflow happens once the exponent argument exceeds $\ln(65504)\approx 11.09$).

### Practice
2. In the Implementation cell, change `d_model` from 32 to 128 and re-run the pre-norm vs.
   post-norm magnitude/gradient comparison at `n_layers=20`. Does a wider model change the
   qualitative pattern (pre-norm grows, post-norm stays controlled), or just the specific numbers?

### Challenge
3. Modify the pre-norm block to scale each sublayer's contribution by $1/\sqrt{n\_layers}$
   before adding it to the residual stream (a simple depth-dependent scaling sometimes used in
   practice), and re-measure the final residual-stream magnitude at `n_layers=40`. Does this
   successfully control the growth while still resembling a pre-norm architecture? Explain the
   tradeoff versus switching to post-norm entirely.

---

## 📚 Further Reading
- Xiong et al., ["On Layer Normalization in the Transformer Architecture"](https://arxiv.org/abs/2002.04745) (pre-norm vs. post-norm analysis)
- Vaswani et al., ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762)
- Dao et al., ["FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness"](https://arxiv.org/abs/2205.14135) (numerically stable, fused attention in practice)

---

*Next notebook: [Combinatorics and Counting](../07_Discrete_Mathematics/01_combinatorics_and_counting.ipynb)*